# AMR Shield — Model Training Pipeline

This notebook trains three classifiers (Logistic Regression, Random Forest, XGBoost)
on the Kaggle Disease-Symptom dataset and saves all artefacts to `backend/models/`.

**Run order:** Execute cells top-to-bottom. Do not skip cells.

**Prerequisites:**
- Place the raw CSV file in `data/raw/` before running.
- All dependencies in `requirements.txt` must be installed.

## 1. Imports and Path Setup

In [ ]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Resolve paths relative to the notebook location so the notebook works
# regardless of where Jupyter is launched from.
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)  # amr-shield/

RAW_DATA_DIR       = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR         = os.path.join(PROJECT_ROOT, 'backend', 'models')

# Ensure output directories exist
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data dir : {RAW_DATA_DIR}')
print(f'Models dir   : {MODELS_DIR}')

## 2. Load Raw Dataset

In [ ]:
# ---------------------------------------------------------------------------
# Locate the CSV file in data/raw/. The Kaggle dataset is typically named
# 'dataset.csv' or 'Disease_symptom_and_patient_profile_dataset.csv'.
# We search for any .csv file in the raw directory so the notebook is
# robust to minor filename differences.
# ---------------------------------------------------------------------------
csv_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError(
        f'No CSV file found in {RAW_DATA_DIR}. '
        'Download the Kaggle Disease-Symptom dataset and place it there.'
    )

csv_path = os.path.join(RAW_DATA_DIR, csv_files[0])
print(f'Loading: {csv_path}')

df_raw = pd.read_csv(csv_path)
print(f'Shape  : {df_raw.shape}')
df_raw.head()

## 3. Inspect and Identify Columns

In [ ]:
print('Columns:', df_raw.columns.tolist())
print('\nDtypes:')
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())

## 4. Clean: Remove Duplicates

In [ ]:
df = df_raw.copy()

rows_before = len(df)

# NOTE: Do NOT call dropna() here.
# This dataset uses a wide format: each row has 1-17 symptoms spread across
# Symptom_1..Symptom_17 columns. Rows with fewer than 17 symptoms have NaN
# in the remaining slots — those NaN slots are NOT missing rows, they are
# valid rows with fewer symptoms. dropna() would delete most of the dataset.
# NaN symptom slots are handled correctly in Cell 5 during the melt step.

# Drop rows where Disease is missing (truly invalid rows)
df.dropna(subset=['Disease'], inplace=True)
print(f'Rows removed (missing Disease) : {rows_before - len(df)}')

rows_after_null = len(df)

# Drop exact duplicate rows
df.drop_duplicates(inplace=True)
print(f'Rows removed (duplicates)      : {rows_after_null - len(df)}')
print(f'Rows remaining                 : {len(df)}')

df.head()

## 5. Feature Engineering: Wide → Binary Matrix

In [ ]:
# ── Step 1: Strip whitespace from Disease column ──────────────────────────────
df['Disease'] = df['Disease'].str.strip()

# ── Step 2: Identify all symptom columns ─────────────────────────────────────
symptom_cols = [col for col in df.columns if col.startswith('Symptom_')]
print(f'Symptom columns found: {symptom_cols}')

# ── Step 3: Attach original row index before melting ─────────────────────────
# Critical — we need row identity to build the binary matrix correctly.
df['row_idx'] = range(len(df))

# ── Step 4: Melt all symptom columns into a single long column ────────────────
melted = df.melt(
    id_vars=['Disease', 'row_idx'],
    value_vars=symptom_cols,
    var_name='symptom_position',
    value_name='symptom'
)

# ── Step 5: Drop NaN symptom entries ONLY (do not drop rows) ─────────────────
# A row with 3 symptoms has 14 NaN symptom slots — drop those NaN slots,
# not the row itself.
melted = melted.dropna(subset=['symptom'])

# ── Step 6: Normalize all symptom strings ────────────────────────────────────
# Order matters:
#   1. strip()  removes leading/trailing whitespace  (fixes ' skin_rash')
#   2. lower()  standardizes case
#   3. replace \s+ with _ fixes internal spaces      (fixes 'dischromic _patches')
melted['symptom'] = (
    melted['symptom']
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
)

# ── Step 7: Drop duplicate (row_idx, symptom) pairs ──────────────────────────
melted = melted.drop_duplicates(subset=['row_idx', 'symptom'])

# ── Step 8: Add binary value column ──────────────────────────────────────────
melted['value'] = 1

# ── Step 9: Pivot to binary feature matrix ───────────────────────────────────
# Rows    = original disease rows (identified by row_idx)
# Columns = one column per unique symptom
# Values  = 1 if that symptom was present in that row, 0 otherwise
binary_matrix = melted.pivot_table(
    index=['row_idx', 'Disease'],
    columns='symptom',
    values='value',
    aggfunc='max',
    fill_value=0
).reset_index()

# ── Step 10: Clean up ─────────────────────────────────────────────────────────
binary_matrix.columns.name = None

# ── Step 11: Separate features (X_raw) and target (y_raw) ────────────────────
X_raw  = binary_matrix.drop(columns=['row_idx', 'Disease']).astype(int)
y_raw  = binary_matrix['Disease']

# ── Verification prints ───────────────────────────────────────────────────────
print(f'\n── Feature Engineering Complete ──')
print(f'Binary matrix shape      : {binary_matrix.shape}')
print(f'Feature columns (X_raw)  : {X_raw.shape[1]} unique symptoms')
print(f'Target rows (y_raw)      : {len(y_raw)}')
print(f'Unique disease classes   : {y_raw.nunique()}')
print(f'\nAll symptom feature columns ({X_raw.shape[1]} total):')
print(sorted(list(X_raw.columns)))
print(f'\nAll disease classes ({y_raw.nunique()} total):')
print(sorted(y_raw.unique()))

## 6. Encode, Scale and Save Artifacts

In [ ]:
print(f'Models will be saved to: {MODELS_DIR}')

# ── Step 1: Encode target labels ──────────────────────────────────────────────
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print(f'\nLabel encoding complete.')
print(f'Total classes : {len(label_encoder.classes_)}')
print(f'Classes       : {list(label_encoder.classes_)}')

# ── Step 2: Scale features ────────────────────────────────────────────────────
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'\nStandard scaling complete.')
print(f'Scaled feature matrix shape: {X.shape}')

# ── Step 3: Capture feature column list ───────────────────────────────────────
# This list defines the exact column ORDER the preprocessor must use
# during live inference. The order must never change after saving.
feature_columns = list(X_raw.columns)
print(f'\nFeature column count : {len(feature_columns)}')
print(f'First 10 features    : {feature_columns[:10]}')

# ── Step 4: Stratified 80/20 train-test split ─────────────────────────────────
# random_state=42 for full reproducibility across all 3 models.
# stratify=y ensures all disease classes appear in both train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'\nTrain samples : {X_train.shape[0]}')
print(f'Test  samples : {X_test.shape[0]}')
print(f'Unique classes in y_train: {len(set(y_train))}')
print(f'Unique classes in y_test : {len(set(y_test))}')

# ── Step 5: Save all preprocessing artifacts ──────────────────────────────────
# preprocessor.py loads: label_encoder.pkl, scaler.pkl, features.pkl
# ml_service.py  loads: logistic_regression.pkl, random_forest.pkl, xgboost.pkl

artifacts = {
    'label_encoder.pkl' : label_encoder,
    'scaler.pkl'         : scaler,
    'features.pkl'       : feature_columns,
}

print('\nSaving artifacts...')
for filename, obj in artifacts.items():
    save_path = os.path.join(MODELS_DIR, filename)
    with open(save_path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  Saved: {save_path}')

print('\n── Cell 6 Complete ──────────────────────────────────────────────────')
print('Artifacts saved  : label_encoder.pkl, scaler.pkl, features.pkl')
print('Variables ready  : X_train, X_test, y_train, y_test')
print('Next step        : Run Cell 7 onwards to train all 3 models')

## 7. Train Logistic Regression

In [ ]:
print('Training Logistic Regression...')

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print('\n--- Logistic Regression: Classification Report ---')
print(classification_report(
    y_test, y_pred_lr,
    target_names=label_encoder.classes_
))

## 8. Train Random Forest

In [ ]:
print('Training Random Forest...')

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print('\n--- Random Forest: Classification Report ---')
print(classification_report(
    y_test, y_pred_rf,
    target_names=label_encoder.classes_
))

## 9. Train XGBoost

In [ ]:
print('Training XGBoost...')

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print('\n--- XGBoost: Classification Report ---')
print(classification_report(
    y_test, y_pred_xgb,
    target_names=label_encoder.classes_
))

## 10. 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_for_cv = [
    ('Logistic Regression', lr_model),
    ('Random Forest',       rf_model),
    ('XGBoost',             xgb_model),
]

print('5-Fold Stratified Cross-Validation (F1 macro)\n')
print(f'{"Model":<25} {"Mean F1":>10} {"Std F1":>10}')
print('-' * 47)

for name, model in models_for_cv:
    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1
    )
    print(f'{name:<25} {scores.mean():>10.4f} {scores.std():>10.4f}')

## 11. Save Trained Models

In [ ]:
lr_path  = os.path.join(MODELS_DIR, 'logistic_regression.pkl')
rf_path  = os.path.join(MODELS_DIR, 'random_forest.pkl')
xgb_path = os.path.join(MODELS_DIR, 'xgboost.pkl')

with open(lr_path, 'wb') as f:
    pickle.dump(lr_model, f)

with open(rf_path, 'wb') as f:
    pickle.dump(rf_model, f)

with open(xgb_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print('All pkl files saved successfully.')
print(f'  {lr_path}')
print(f'  {rf_path}')
print(f'  {xgb_path}')

## 12. Write Model Versioning Metadata

In [ ]:
import sklearn
import xgboost
import sys
sys.path.insert(0, PROJECT_ROOT)

from backend.services.model_metadata import write_metadata

# Collect mean CV F1 scores computed in cell 12
cv_scores = {}
for name, model in models_for_cv:
    s = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_scores[name] = round(float(s.mean()), 4)

write_metadata(
    dataset_filename=os.path.basename(csv_path),
    dataset_rows=len(df),
    num_features=len(feature_columns),
    num_classes=len(label_encoder.classes_),
    sklearn_version=sklearn.__version__,
    xgboost_version=xgboost.__version__,
    model_scores=cv_scores,
)

print('model_metadata.json written to backend/models/')
print(f'  Trained at    : {__import__("datetime").datetime.now()}')
print(f'  Dataset       : {os.path.basename(csv_path)}')
print(f'  Rows          : {len(df)}')
print(f'  Features      : {len(feature_columns)}')
print(f'  Classes       : {len(label_encoder.classes_)}')
print(f'  sklearn       : {sklearn.__version__}')
print(f'  xgboost       : {xgboost.__version__}')
print(f'  CV F1 scores  : {cv_scores}')

## 13. Verify All Artifacts Exist

In [ ]:
expected_files = [
    'label_encoder.pkl',
    'scaler.pkl',
    'features.pkl',
    'logistic_regression.pkl',
    'random_forest.pkl',
    'xgboost.pkl',
    'model_metadata.json',
]

print('Artifact verification:')
all_present = True
for filename in expected_files:
    path = os.path.join(MODELS_DIR, filename)
    exists = os.path.isfile(path)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {filename}')
    if not exists:
        all_present = False

if all_present:
    print('\nAll 7 artifact files confirmed. Training pipeline complete.')
else:
    print('\nWARNING: One or more artifact files are missing. Re-run the notebook.')